In [11]:
%pip -q install gradio pandas scikit-learn nltk openpyxl joblib

In [12]:
import re
import traceback
import joblib
import pandas as pd
import gradio as gr

from pathlib import Path
from datetime import datetime

from nltk.stem import SnowballStemmer

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)
from google.colab import drive, files

# Connect Google Drive
drive.mount('/content/drive')

# Create project folders
PROJECT_DIR = Path(
    '/content/drive/MyDrive/StudentFeedbackAnalysis'
)

MODEL_DIR = PROJECT_DIR / 'models'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
# Excel file location
FEEDBACK_FILE = PROJECT_DIR / 'student_feedback.xlsx'

# Excel columns
FEEDBACK_COLUMNS = [
    'Timestamp',
    'Feedback',
    'Predicted Sentiment',
    'Sentiment Model Score',
    'Predicted Category',
    'Category Model Score'
]
# Create Excel workbook if it doesn't exist
if not FEEDBACK_FILE.exists():

    pd.DataFrame(
        columns=FEEDBACK_COLUMNS
    ).to_excel(
        FEEDBACK_FILE,
        index=False,
        engine='openpyxl'
    )
print("Google Drive connected successfully!")
print("Excel location:", FEEDBACK_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive connected successfully!
Excel location: /content/drive/MyDrive/StudentFeedbackAnalysis/student_feedback.xlsx


In [13]:
uploaded = files.upload()

csv_name = next(
    (
        name for name in uploaded
        if name.lower().endswith('.csv')
    ),
    None
)
if csv_name is None:
    raise ValueError("Please upload feedback_dataset.csv")
df = pd.read_csv(csv_name)
required_columns = [
    'feedback',
    'sentiment',
    'category'
]
missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")
df = df[required_columns].dropna().drop_duplicates().copy()
for col in required_columns:
    df[col] = df[col].astype(str).str.strip()


valid_sentiments = {
    'Positive', 'Negative', 'Neutral'
}
valid_categories = {
    'Teachers',
    'Classrooms',
    'Hostel',
    'University Facilities'
}
if not set(df['sentiment']).issubset(valid_sentiments):
    raise ValueError("Invalid sentiment labels.")

if not set(df['category']).issubset(valid_categories):
    raise ValueError("Invalid category labels.")

if len(df) < 30:
    raise ValueError("Dataset is too small.")


print("Dataset loaded successfully!")
print("Total feedback records:", len(df))

display(df.head(10))

print("\nSentiment Distribution:")
print(df['sentiment'].value_counts())

print("\nCategory Distribution:")
print(df['category'].value_counts())

Saving feedback_dataset.csv to feedback_dataset (1).csv
Dataset loaded successfully!
Total feedback records: 420


,feedback,sentiment,category
0,I want to ask about drinking water facilities;...,Neutral,University Facilities
1,The lecture halls are spacious and the fans wo...,Positive,Classrooms
2,I have had a poor experience with the lecture ...,Negative,Classrooms
3,I appreciate the lecture rooms because they ha...,Positive,Classrooms
4,I have a question regarding the teaching space...,Neutral,Classrooms
5,I want to ask about the hostel rooms; their vi...,Neutral,Hostel
6,I am disappointed with campus transport facili...,Negative,University Facilities
7,I am happy with the teaching spaces; they are ...,Positive,Classrooms
8,The sports centre registration form is availab...,Neutral,University Facilities
9,Maintenance requests can be submitted at the r...,Neutral,Hostel



Sentiment Distribution:
sentiment
Neutral     140
Positive    140
Negative    140
Name: count, dtype: int64

Category Distribution:
category
University Facilities    105
Classrooms               105
Hostel                   105
Teachers                 105
Name: count, dtype: int64


In [14]:
# NLP preprocessing

stemmer = SnowballStemmer('english')

stop_words = set(ENGLISH_STOP_WORDS) - {
    'no', 'not', 'nor', 'never'
}
def preprocess_text(text):
    words = re.findall(
        r'[a-z]+',
        str(text).lower()
    )
    words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]
    return ' '.join(words)
# Create Machine Learning pipeline

def create_model():

    return Pipeline([

        ('tfidf', TfidfVectorizer(
            preprocessor=preprocess_text,
            lowercase=False,
            ngram_range=(1, 2)
        )),

        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        ))

    ])
# Prepare data

X = df['feedback']

y_sentiment = df['sentiment']

y_category = df['category']


# Split training and testing data

stratify_labels = (
    y_sentiment.astype(str)
    + '|'
    + y_category.astype(str)
)
X_train, X_test, ys_train, ys_test, yc_train, yc_test = train_test_split(
    X,
    y_sentiment,
    y_category,
    test_size=0.20,
    random_state=42,
    stratify=stratify_labels
)
# Create two models

sentiment_model = create_model()
category_model = create_model()


# Train both models

sentiment_model.fit(X_train, ys_train)

category_model.fit(X_train, yc_train)


# Predict test data

sentiment_pred = sentiment_model.predict(X_test)
category_pred = category_model.predict(X_test)
# Evaluate models

def evaluate_model(actual, predicted):

    precision, recall, f1, _ = precision_recall_fscore_support(
        actual,
        predicted,
        average='weighted',
        zero_division=0
    )
    return {
        'Accuracy': accuracy_score(actual, predicted),
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1
    }
results = pd.DataFrame(
    [
        evaluate_model(ys_test, sentiment_pred),
        evaluate_model(yc_test, category_pred)
    ],
    index=[
        'Sentiment Model',
        'Category Model'
    ]
)
display(results.round(3))
# Save trained models in Google Drive

joblib.dump(
    sentiment_model,
    MODEL_DIR / 'sentiment_model.pkl'
)

joblib.dump(
    category_model,
    MODEL_DIR / 'category_model.pkl'
)


print("Both models trained and saved successfully!")

,Accuracy,Precision,Recall,F1-score
Sentiment Model,0.905,0.912,0.905,0.905
Category Model,1.000,1.000,1.000,1.000


Both models trained and saved successfully!


In [8]:
import gradio as gr
import pandas as pd
import threading
import traceback

from pathlib import Path
from datetime import datetime
# Google Drive storage location

PROJECT_DIR = Path(
    "/content/drive/MyDrive/StudentFeedbackAnalysis"
)

PROJECT_DIR.mkdir(parents=True, exist_ok=True)


CSV_FILE = PROJECT_DIR / "student_feedback.csv"

EXCEL_FILE = PROJECT_DIR / "student_feedback.xlsx"
COLUMNS = [
    "Timestamp",
    "Feedback",
    "Predicted Sentiment",
    "Sentiment Confidence",
    "Predicted Category",
    "Category Confidence"
]

# Prevent simultaneous file-writing conflicts

save_lock = threading.Lock()
# Automatically save feedback

def save_feedback(record):

    with save_lock:

        new_row = pd.DataFrame(
            [record],
            columns=COLUMNS
        )

        # Keep previous Excel records if CSV is not
        # present yet and the old workbook has data.

        if not CSV_FILE.exists() and EXCEL_FILE.exists():

            try:
                old_data = pd.read_excel(EXCEL_FILE)

                old_data = old_data.reindex(
                    columns=COLUMNS
                )

                if not old_data.empty:

                    old_data.to_csv(
                        CSV_FILE,
                        index=False,
                        encoding="utf-8-sig"
                    )

            except Exception as error:

                print("Previous workbook import:", error)

        # Save the new record permanently to CSV
        new_row.to_csv(
            CSV_FILE,
            mode="a",
            header=not CSV_FILE.exists(),
            index=False,
            encoding="utf-8-sig"
        )

        # Rebuild Excel using ALL saved records

        all_records = pd.read_csv(CSV_FILE)

        all_records.to_excel(
            EXCEL_FILE,
            index=False,
            engine="openpyxl"
        )

        return len(all_records)
# Feedback submission function

def submit_feedback(text):

    try:

        text = (text or "").strip()

        if len(text) < 8:

            return (
                "Please enter at least 8 characters.",
                "",
                "",
                "No feedback saved."
            )

        if len(text) > 1500:

            return (
                "Feedback must be under 1500 characters.",
                "",
                "",
                "No feedback saved."
            )
        # AI predictions

        sentiment = sentiment_model.predict([text])[0]
        category = category_model.predict([text])[0]

        # Prediction confidence

        sentiment_score = float(
            sentiment_model.predict_proba([text]).max()
        )

        category_score = float(
            category_model.predict_proba([text]).max()
        )


        # Create feedback record

        record = {

            "Timestamp": datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

            "Feedback": text,

            "Predicted Sentiment": sentiment,

            "Sentiment Confidence": round(
                sentiment_score, 4
            ),

            "Predicted Category": category,

            "Category Confidence": round(
                category_score, 4
            )
        }


        # Save and verify record count

        total_saved = save_feedback(record)


        return (

            "✅ Feedback successfully analysed and saved!",

            f"{sentiment} ({sentiment_score:.1%})",

            f"{category} ({category_score:.1%})",

            f"Total feedback records saved: {total_saved}"

        )


    except Exception as error:

        traceback.print_exc()

        return (
            f"❌ Saving failed: {type(error).__name__}: {error}",
            "",
            "",
            "Please report this error to the administrator."
        )


# Close old Gradio application if one exists

try:

    if "demo" in globals():
        demo.close()

except Exception:
    pass


# Create new interface

with gr.Blocks(
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        "# Student Feedback Analysis System"
    )

    gr.Markdown(
        "Submit your university feedback. "
        "Our Machine Learning models will analyse it."
    )


    with gr.Row():

        with gr.Column(scale=3):

            feedback_input = gr.Textbox(

                label="Enter Your Feedback",

                placeholder=(
                    "Share your experience about "
                    "teachers, classrooms, hostel "
                    "or university facilities..."
                ),

                lines=5,

                max_lines=8
            )


            submit_button = gr.Button(

                "Analyse Feedback",

                variant="primary"
            )


        with gr.Column(scale=2):

            gr.Markdown("### AI Analysis Results")


            status_output = gr.Markdown(
                "Your results will appear here."
            )

            sentiment_output = gr.Textbox(
                label="Predicted Sentiment",
                interactive=False
            )

            category_output = gr.Textbox(
                label="Predicted Category",
                interactive=False
            )

            count_output = gr.Markdown(
                "**Total submissions: 0**"
            )

    gr.Markdown(
        "Your feedback is stored anonymously. "
        "Please do not enter names, roll numbers "
        "or personal information."
    )

    submit_button.click(

        fn=submit_feedback,

        inputs=feedback_input,

        outputs=[
            status_output,
            sentiment_output,
            category_output,
            count_output
        ]

    )


# Keep the portal responsive

demo.queue(default_concurrency_limit=1)
demo.launch(

    share=True,
    prevent_thread_lock=True,
    show_error=True

)
print("Student Feedback Portal is running!")
print("Feedback storage:", CSV_FILE)
print("Excel workbook:", EXCEL_FILE)

Closing server running on port: 7860


/tmp/ipykernel_1191/3644164697.py:212: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8a7e3686ca263b12bf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Student Feedback Portal is running!
Feedback storage: /content/drive/MyDrive/StudentFeedbackAnalysis/student_feedback.csv
Excel workbook: /content/drive/MyDrive/StudentFeedbackAnalysis/student_feedback.xlsx


In [15]:
from google.colab import files
import pandas as pd

if CSV_FILE.exists():
    saved_records = pd.read_csv(CSV_FILE)
    print("Total saved feedback:", len(saved_records))
    print("\nLatest student submissions:")
    display(saved_records.tail(10))
    # Rebuild the latest Excel file
    saved_records.to_excel(
        EXCEL_FILE,
        index=False,
        engine="openpyxl"
    )
    print("\nDownloading updated Excel workbook...")

    files.download(str(EXCEL_FILE))
else:
    print("No student feedback has been submitted yet.")

Total saved feedback: 3

Latest student submissions:


,Timestamp,Feedback,Predicted Sentiment,Sentiment Confidence,Predicted Category,Category Confidence
0,2026-11-25 08:26:21,I want to ask about drinking water facilities,Neutral,NaN,University Facilities,NaN
1,2026-11-25 08:32:14,I want to ask about drinking water facilities,Neutral,0.4585,University Facilities,0.6797
2,2026-11-25 08:33:50,Maintenance requests can be submitted at the r...,Neutral,0.6023,Hostel,0.5078


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>